# Sesi 3 — Knowledge Agent: Prompting ReAct (Qwen Lokal)

Bagian dari kurikulum **Agentic Knowledge System — FastAPI + DuckDB + Qwen (Local, tanpa LLM cloud)**.

> Jalankan sel dari atas ke bawah secara berurutan. Sel bertanda `🔨 TODO` wajib dilengkapi peserta; solusi contoh disediakan pada sel setelahnya (bisa disembunyikan saat sesi diberikan ke peserta).

## 0. Setup Environment
Kita pakai `llama-cpp-python` untuk menjalankan Qwen 2.5 GGUF secara lokal di Colab. Unduh model butuh koneksi internet penuh (jalankan langsung di Colab, bukan di sandbox lain).

In [ ]:
!pip -q install llama-cpp-python duckdb sentence-transformers fastapi httpx

## 1. Unduh Model Qwen 2.5 (GGUF)
Pakai varian kecil (misal `Qwen2.5-0.5B-Instruct-GGUF`) agar cepat di Colab CPU/GPU gratis.

### Opsi Download Model
Pilih salah satu sumber:
- **Google Drive (direkomendasikan, lebih cepat):** [Download dari Google Drive](https://drive.google.com/drive/folders/16eYzbAx7KOnawHqmnMD6tjshSSCmp6sX?usp=sharing)
- **Hugging Face (alternatif):** [Qwen2.5-0.5B-Instruct-GGUF di Hugging Face](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF)

Setelah download dari Google Drive, upload file `.gguf` ke Colab atau mount Google Drive, lalu sesuaikan `MODEL_PATH` di sel berikut.

In [ ]:
import os

MODEL_PATH = "qwen2.5-0.5b-instruct-q4_k_m.gguf"

if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} "https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf"
print("Model siap:", os.path.exists(MODEL_PATH))

In [ ]:
from llama_cpp import Llama

llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_threads=4, verbose=False)

def qwen_llm_call(prompt, stop=None):
    out = llm(prompt, max_tokens=256, stop=stop or ["Observation:"])
    return out["choices"][0]["text"]

## 2. Siapkan Ulang Knowledge Base & API (dari Sesi 1-2)

In [ ]:
import duckdb, uuid
from sentence_transformers import SentenceTransformer
from fastapi import FastAPI
from fastapi.testclient import TestClient

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
con = duckdb.connect("knowledge.duckdb")
con.execute("INSTALL vss; LOAD vss;")
con.execute("""
CREATE TABLE IF NOT EXISTS documents (
    id VARCHAR PRIMARY KEY, source VARCHAR, content TEXT,
    embedding FLOAT[384], created_at TIMESTAMP DEFAULT current_timestamp
);
""")
if con.execute("SELECT COUNT(*) FROM documents").fetchone()[0] == 0:
    seed = [("faq_produk.txt", "Garansi produk berlaku 1 tahun sejak tanggal pembelian."),
            ("sop_retur.txt", "Retur barang maksimal 7 hari setelah barang diterima.")]
    for source, text in seed:
        emb = embed_model.encode(text).tolist()
        con.execute("INSERT INTO documents VALUES (?, ?, ?, ?, current_timestamp)",
                    [str(uuid.uuid4()), source, text, emb])

app = FastAPI()

@app.get("/documents/search")
def search_documents(q: str, k: int = 5):
    q_emb = embed_model.encode(q).tolist()
    rows = con.execute("""
        SELECT id, source, content, array_distance(embedding, ?::FLOAT[384]) AS dist
        FROM documents ORDER BY dist ASC LIMIT ?
    """, [q_emb, k]).fetchall()
    return [{"id": r[0], "source": r[1], "content": r[2], "score": r[3]} for r in rows]

client = TestClient(app)

## 3. Prompt Template ReAct

In [ ]:
REACT_SYSTEM = """Kamu adalah agent yang menjawab pertanyaan memakai tool.
Tool tersedia:
- search_knowledge[query]: mencari dokumen relevan di knowledge base.

Format WAJIB setiap langkah:
Thought: <pemikiranmu>
Action: search_knowledge
Action Input: <query pencarian>

Jika sudah cukup informasi:
Thought: <pemikiran akhir>
Final Answer: <jawaban ke user>
"""

## 4. Tool Executor & ReAct Loop

In [ ]:
def call_tool(action, action_input):
    if action == "search_knowledge":
        r = client.get("/documents/search", params={"q": action_input, "k": 3})
        return r.json()
    return {"error": "unknown tool"}

import re

def react_loop(query, max_steps=4, verbose=True):
    history = f"{REACT_SYSTEM}\nQuestion: {query}\n"
    for step in range(max_steps):
        output = qwen_llm_call(history)
        if verbose:
            print(f"--- Step {step} ---\n{output}\n")
        if "Final Answer:" in output:
            return output.split("Final Answer:")[-1].strip()

        action_match = re.search(r"Action:\s*(\w+)", output)
        input_match = re.search(r"Action Input:\s*(.+)", output)
        if not action_match or not input_match:
            history += output + "\nThought: Format salah, saya ulangi.\n"
            continue

        observation = call_tool(action_match.group(1).strip(), input_match.group(1).strip())
        history += output + f"\nObservation: {observation}\n"
    return "Maaf, saya belum menemukan jawaban dalam batas langkah."

In [ ]:
answer = react_loop("Berapa lama masa garansi produk?")
print("FINAL:", answer)

### 🔨 TODO 1
Tambahkan tool kedua `create_document[content]` yang memanggil endpoint POST `/documents` (perlu menambahkan endpoint tersebut ke `app` di atas terlebih dahulu). Update `REACT_SYSTEM` dan `call_tool` agar mendukung tool baru ini.

In [ ]:
# TODO: implementasikan tool create_document
# 1. tambahkan endpoint POST /documents di app
# 2. update REACT_SYSTEM dengan deskripsi tool baru
# 3. update call_tool untuk menangani action == "create_document"


### 🔨 TODO 2
Jalankan `react_loop` untuk 5 pertanyaan berbeda, catat berapa kali model gagal format (masuk ke cabang "Format salah"). Hitung success rate-nya.

In [ ]:
test_questions = [
    "Apa syarat retur barang?",
    "____",
    "____",
]
for q in test_questions:
    print(q, "->", react_loop(q, verbose=False))

## Ringkasan Sesi 3
Agent lokal berbasis Qwen sudah bisa melakukan ReAct loop dengan tool `search_knowledge`. Sesi 4 akan membandingkan pendekatan iteratif ini dengan pola *planner-executor* (structured JSON prompting) memakai model Qwen yang sama.